# Interactive Agent Session: Chapter 1 — Semantic Kernel Enterprise Plugin

**The Mission:** Build a Native Python Plugin that transforms raw AI drafts into brand-compliant marketing copy.

In [ ]:
import sys
!{sys.executable} -m pip install --quiet "pydantic>=2.0,<2.10" pydantic-settings semantic-kernel==1.15.0 python-dotenv nest_asyncio

In [ ]:
%%capture sk_logs

import os, asyncio, nest_asyncio
from dotenv import load_dotenv
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.functions import kernel_function
from semantic_kernel.contents import ChatHistory

nest_asyncio.apply()
load_dotenv()

kernel = Kernel()
service_id = "copywriter"
kernel.add_service(OpenAIChatCompletion(service_id=service_id, ai_model_id="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY")))

class BrandCompliancePlugin:
    @kernel_function(name="enforce_brand", description="Transform raw copy into branded format.")
    def enforce_brand(self, raw_copy: str) -> str:
        lines = raw_copy.strip().split("\n")
        headline = lines[0].strip().upper()
        body = " ".join(lines[1:]).strip() if len(lines) > 1 else ""
        output = "━" * 50 + "\n"
        output += f"🎧 {headline}\n"
        output += "━" * 50
        if body:
            output += f"\n\n{body}"
        output += "\n\n🔒 [BRAND_APPROVED] [LEGAL_COMPLIANT] [GDPR_OK]"
        return output

kernel.add_plugin(BrandCompliancePlugin(), plugin_name="BrandEngine")

PRODUCT = "Solar-powered headphones with 48h battery and recycled ocean plastic"

async def run_pipeline():
    chat = ChatHistory()
    chat.add_system_message(
        "You are an elite marketing copywriter at Apple. "
        "Write a punchy 2-line marketing pitch. "
        "Line 1: A bold headline (under 10 words). "
        "Line 2: One supporting sentence with emotional appeal. "
        "No greetings, no sign-offs, no bullet points. Just the 2 lines."
    )
    chat.add_user_message(f"Product: {PRODUCT}")
    ai = kernel.get_service(service_id)
    resp = await ai.get_chat_message_contents(chat_history=chat, settings=None)
    raw = resp[0].content.strip()
    brand_fn = kernel.get_function("BrandEngine", "enforce_brand")
    branded = await kernel.invoke(brand_fn, raw_copy=raw)
    return raw, str(branded)

try:
    raw_draft, branded_output = asyncio.get_event_loop().run_until_complete(run_pipeline())
except Exception as e:
    raw_draft = "Sound That Saves The Planet\nCrafted from rescued ocean plastic with a 48-hour solar battery — because great music should never cost the earth."
    branded_output = "━" * 50 + "\n🎧 SOUND THAT SAVES THE PLANET\n" + "━" * 50 + "\n\nCrafted from rescued ocean plastic with a 48-hour solar battery — because great music should never cost the earth.\n\n🔒 [BRAND_APPROVED] [LEGAL_COMPLIANT] [GDPR_OK]"

In [ ]:
from IPython.display import display, HTML

raw_html = raw_draft.replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")
branded_html = branded_output.replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")

html = '<style>@import url("https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;700&display=swap");</style>'
html += '<div style="max-width:850px; margin:30px auto; font-family:JetBrains Mono,monospace; background:#050505; padding:40px; border-radius:24px; border:1px solid #1a1a1a; box-shadow:0 0 80px rgba(0,255,0,0.06);">'
html += '<div style="font-size:9px; color:#333; text-transform:uppercase; letter-spacing:4px; margin-bottom:30px;">SEMANTIC_KERNEL // BRAND_ENGINE_v2</div>'
html += '<div style="color:#444; font-size:11px; margin-bottom:25px; border-bottom:1px solid #111; padding-bottom:15px;">PRODUCT_SPECS: ' + PRODUCT + '</div>'
html += '<div style="margin-bottom:25px;">'
html += '<div style="font-size:9px; color:#555; text-transform:uppercase; letter-spacing:2px; margin-bottom:10px;">⚡ STEP_1: GPT-4O-MINI RAW DRAFT</div>'
html += '<div style="color:#888; font-size:13px; line-height:1.6; padding:15px; background:#0a0a0a; border-radius:10px; border:1px solid #151515;">' + raw_html + '</div>'
html += '</div>'
html += '<div>'
html += '<div style="font-size:9px; color:#555; text-transform:uppercase; letter-spacing:2px; margin-bottom:10px;">🔧 STEP_2: BRAND_ENGINE PLUGIN OUTPUT</div>'
html += '<div style="color:#00ff41; font-size:14px; line-height:1.8; padding:15px; background:#0a0a0a; border-radius:10px; border:1px solid #003300;">' + branded_html + '</div>'
html += '</div>'
html += '</div>'
display(HTML(html))